In [1]:
!pip install pyspark

**widgets**

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
   .appName("widgets") \
   .getOrCreate()

In [3]:
from google.colab import files, output
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:", len(df))

display(df.head())

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


**Transformation and Actions**

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
   .appName("transformation_actions") \
   .getOrCreate()

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [9]:
filter_df = df.filter(col("quantity") > 3)

print("Transformation created")

selected_df = filter_df.select(
    "city", "product", "quantity", "price"
)

print("Another transformation created")

selected_df.show()

Transformation created
Another transformation created
+---------+-------+--------+-----+
|     city|product|quantity|price|
+---------+-------+--------+-----+
|Bangalore| Mobile|       5|20000|
|Hyderabad| Mobile|       4|22000|
|Bangalore| Tablet|       6|12000|
|   Mumbai| Mobile|       8|18000|
|Hyderabad| Tablet|       5|14000|
|   Mumbai| Tablet|       7|13000|
|  Chennai| Mobile|       6|21000|
+---------+-------+--------+-----+



In [10]:
step1 = df.filter(col("quantity") > 3)
step2 = step1.withColumn("revenue", col("quantity") * col("price"))
step3 = step2.groupBy("city").agg(
    sum("quantity").alias("total_quantity")
)
step4 = step3.orderBy(desc("total_quantity"))
print(" All Transformation defined")

step4.show()

 All Transformation defined
+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|   Mumbai|            15|
|Bangalore|            11|
|Hyderabad|             9|
|  Chennai|             6|
+---------+--------------+



In [12]:
result = df.filter(col("price") > 20000) \
           .select("city","product","price") \
           .orderBy(desc("price"))
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=109]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [13]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [14]:
high_value_sales =df.filter(col("price") > 20000)
high_value_sales.show()
print("total records:", high_value_sales.count())
records = high_value_sales.collect()
print(records)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

total records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]
